# Proyecto — Data Stream Processor
## Contexto:
La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.

## Objetivo
Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases ArrayStack y ArrayQueue proporcionadas por el curso.

### Atributos en __init__:
- Queue: registros pendientes (FIFO).
- Stack: historial de cambios realizados (LIFO).
- Lista: registros actuales.

In [ ]:
from goodrich.ch06.array_stack import ArrayStack, Empty
from goodrich.ch06.array_queue import ArrayQueue

In [ ]:
class DataProcessor:

    #   1. Se definen los tres atributos principales sin modificar la lógica interna de ArrayQueue ni ArrayStack.  
    
    def __init__(self):
        self._queue = ArrayQueue()
        self._stack = ArrayStack()
        self._state = []

     #  2. Se valida que el argumento cumpla con la estructura de 3 elementos (sensor, variable, value) y que el value sea un número.
     #  Lanza ValueError cuando no cumple estos parámetros. Si es correcto, lo añade a _queue sin modificar el estado ni el historial.   
    
    def add(self,record):
        if not isinstance(record, (tuple, list)) or len(record) != 3:
            raise ValueError(
                "El registro debe ser una tupla o lista de exactamente 3"
                " elementos: (sensor, variable, value)"
            )

        sensor, variable, value = record
        # Validar que el valor sea numérico
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise ValueError("El valor ('value') del registro debe ser numérico")

        # Agregar el registro a la cola respetando el orden de llegada
        self._queue.enqueue(record)

    #  3. Desencola el siguiente registro respetando el orden FIFO. Antes de actualizar _state, verifica si la combinación (sensor, variable) ya existía
    #   para guardar su valor previo en _stack. Si no existía, guarda en el historial un marcador indicando que la clave es nueva para que undo()
    #   pueda eliminarla si es necesario. Si la cola está vacía, genera la excepción Empty.
    
    def process_next(self):
        if self._queue.is_empty():
            raise Empty("No hay registros pendientes para procesar")

        record = self._queue.dequeue()
        sensor, variable, value = record

        previous_value = None
        existed_before = False
        index_to_update = -1

        for i, item in enumerate(self._state):
            if item[0] == sensor and item[1] == variable:
                existed_before = True
                previous_value = item[2]
                index_to_update = i
                break

        self._stack.push((sensor, variable, existed_before, previous_value))
        if existed_before:
            self._state[index_to_update] = (sensor, variable, value)
        else:
            self._state.append((sensor, variable, value))
        return record

# 4. Extrae del _stack (comportamiento LIFO) la información sobre el último registro modificado o creado.
# Si la variable existía antes de ser procesada, restaura la tupla (sensor, variable, previous_value) en _state.
# Si era una variable totalmente nueva que antes no existía en el sistema, la busca y la remueve de la lista _state.
# Lanza la excepción Empty si _stack está vacío
    
    def undo(self):
        if self._stack.is_empty():
            raise Empty("No hay cambios que deshacer en el historial")

        sensor, variable, existed_before, previous_value = self._stack.pop()

        if existed_before:
            for i, item in enumerate(self._state):
                if item[0] == sensor and item[1] == variable:
                    self._state[i] = (sensor, variable, previous_value)
                    break
        else:
            for i, item in enumerate(self._state):
                if item[0] == sensor and item[1] == variable:
                    self._state.pop(i)
                    break

# 5. Retorna la cantidad de elementos en _queue que aguardan ser procesados.
    def pending(self):
        return len(self._queue)

# 6. Recorre la lista _state buscando coincidencia exacta del par (sensor, variable) y retorna su value.
# Si la combinación no ha sido registrada/procesada, lanza KeyError.

    def current_value(self, sensor, variable):
        for item in self._state:
            if item[0] == sensor and item[1] == variable:
                return item[2]

        raise KeyError(
            f"No existe un valor para el sensor '{sensor}' y variable"
            f" '{variable}'"
        )



## 8. Ejemplo completo de uso
Demostración del flujo con recepción de registros, procesamiento gradual y uso de `undo()`.

In [ ]:
processor = DataProcessor()

A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

processor.add(A)
processor.add(B)
processor.add(C)
print(f"Registros pendientes en cola: {processor.pending()}")

r_a = processor.process_next()
print(f"Procesado {r_a} | Temperatura: {processor.current_value('S01', 'temperature')}")

r_b = processor.process_next()
print(f"Procesado {r_b} | Temperatura: {processor.current_value('S01', 'temperature')}")

r_c = processor.process_next()
print(f"Procesado {r_c} | Humedad: {processor.current_value('S01', 'humidity')}")

processor.undo()
print("\nSe ejecutó undo()...")
try:
    processor.current_value("S01", "humidity")
except KeyError as e:
    print(f"S01/humidity eliminado del estado: {e}")

processor.undo()
print(f"Se ejecutó undo() | Temperatura restaurada: {processor.current_value('S01', 'temperature')}")

processor.undo()
print("Se ejecutó undo()...")
try:
    processor.current_value("S01", "temperature")
except KeyError as e:
    print(f"S01/temperature eliminado del estado: {e}")


## 9. Pruebas obligatorias
Ejecución de los 17 casos de prueba para verificar la solidez de la implementación.

In [ ]:
p = DataProcessor()

#  1. Verifica que pending() retorne 0 en un procesador vacío.
assert p.pending() == 0

#  2. Verifica que add() incremente el contador a 1.
p.add(("S01", "temp", 18.5))
assert p.pending() == 1

#  3. Verifica que múltiples add() se contabilicen correctamente.
p.add(("S01", "temp", 22.0))
p.add(("S02", "hum", 70.0))
assert p.pending() == 3

#  4. Verifica orden FIFO en process_next().
r1 = p.process_next()
assert r1 == ("S01", "temp", 18.5)

#  5. Verifica que process_next() decremente pendientes.
assert p.pending() == 2

#  6. Verifica que procesar todo reduzca pending() a 0.
p.process_next()
p.process_next()
assert p.pending() == 0

#  7. Verifica actualización de variables existentes.
p.add(("S01", "temp", 30.0))
p.process_next()
assert p.current_value("S01", "temp") == 30.0

#  8. Verifica consulta de valores actuales.
assert p.current_value("S02", "hum") == 70.0

#  9. Verifica restaurar valor previo con undo().
p.undo()
assert p.current_value("S01", "temp") == 22.0

# 10. Múltiples undo() consecutivos.
p.undo()
p.undo()
assert p.current_value("S01", "temp") == 18.5

# 11. Lanzamiento de Empty en process_next() con cola vacía.
try:
    p.process_next()
    assert False
except Empty:
    pass

# 12. Lanzamiento de Empty en undo() con historial vacío.
p.undo()
try:
    p.undo()
    assert False
except Empty:
    pass

# 13. Eliminar del estado una variable nueva mediante undo().
p.add(("S99", "press", 1013.25))
p.process_next()
p.undo()
try:
    p.current_value("S99", "press")
    assert False
except KeyError:
    pass

# 14. Encadenamiento de cambios sobre una misma variable.
p.add(("S01", "temp", 10.0))
p.add(("S01", "temp", 15.0))
p.add(("S01", "temp", 20.0))
p.process_next()
p.process_next()
p.process_next()
assert p.current_value("S01", "temp") == 20.0
p.undo()
assert p.current_value("S01", "temp") == 15.0
p.undo()
assert p.current_value("S01", "temp") == 10.0
p.undo()

# 15. Lanzamiento de ValueError si no tiene 3 elementos.
try:
    p.add(("S01", "temp"))
    assert False
except ValueError:
    pass

# 16. Lanzamiento de ValueError si el valor no es numérico.
try:
    p.add(("S01", "temp", "25 C"))
    assert False
except ValueError:
    pass

# 17. Lanzamiento de KeyError si el sensor/variable no existe.
try:
    p.current_value("S01", "no_existe")
    assert False
except KeyError:
    pass

print('¡Las 17 pruebas se ejecutaron exitosamente!')


## 10. Análisis de complejidad temporal

- **`add(record)` — $O(1)$**: Validación en $O(1)$ e inserción al final de `ArrayQueue` (`enqueue`) en $O(1)$ amortizado.
- **`process_next()` — $O(n)$**: Extraer de la cola toma $O(1)$, pero la búsqueda de `(sensor, variable)` en `_state` recorre hasta $n$ elementos.
- **`undo()` — $O(n)$**: Extraer del `ArrayStack` toma $O(1)$, mientras que buscar o eliminar el elemento en `_state` toma $O(n)$.
- **`pending()` — $O(1)$**: Retorna el tamaño directo de la cola.
- **`current_value(sensor, variable)` — $O(n)$**: Realiza un recorrido secuencial en `_state`.

## 11. Decisiones de diseño

1. **Guardado por Deltas en el Historial**: En vez de copiar toda la lista `_state` en cada cambio ($O(n)$ espacio), se apilan únicamente los cambios `(sensor, variable, existed_before, previous_value)`, consumiendo $O(1)$ de espacio por operación.
2. **Estructuras requeridas**: Uso estricto de `ArrayQueue` (FIFO) y `ArrayStack` (LIFO) del curso.
3. **Excepciones**: Manejo preciso de `ValueError`, `KeyError` y `Empty`.

## 12. Bonus — Implementación de `redo()`

Clase `DataProcessorWithRedo` con la pila `_redo_stack` para re-aplicar cambios.

In [ ]:
class DataProcessorWithRedo:

    # 1. Atributos principales + pila auxiliar para redo
    def __init__(self):
        self._queue = ArrayQueue()
        self._stack = ArrayStack()
        self._redo_stack = ArrayStack()
        self._state = []

    # 2. Validación y encolado
    def add(self, record):
        if not isinstance(record, (tuple, list)) or len(record) != 3:
            raise ValueError("El registro debe ser una tupla o lista de exactamente 3 elementos: (sensor, variable, value)")
        sensor, variable, value = record
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise ValueError("El valor ('value') del registro debe ser numérico")
        self._queue.enqueue(record)

    # 3. Procesamiento y limpieza de la pila de redo
    def process_next(self):
        if self._queue.is_empty():
            raise Empty("No hay registros pendientes para procesar")
        record = self._queue.dequeue()
        sensor, variable, value = record

        previous_value = None
        existed_before = False
        index_to_update = -1

        for i, item in enumerate(self._state):
            if item[0] == sensor and item[1] == variable:
                existed_before = True
                previous_value = item[2]
                index_to_update = i
                break

        self._stack.push((sensor, variable, existed_before, previous_value, value))
        self._redo_stack = ArrayStack()

        if existed_before:
            self._state[index_to_update] = (sensor, variable, value)
        else:
            self._state.append((sensor, variable, value))
        return record

    # 4. Deshacer y transferir a redo_stack
    def undo(self):
        if self._stack.is_empty():
            raise Empty("No hay cambios que deshacer en el historial")

        sensor, variable, existed_before, previous_value, current_value = self._stack.pop()
        self._redo_stack.push((sensor, variable, existed_before, previous_value, current_value))

        if existed_before:
            for i, item in enumerate(self._state):
                if item[0] == sensor and item[1] == variable:
                    self._state[i] = (sensor, variable, previous_value)
                    break
        else:
            for i, item in enumerate(self._state):
                if item[0] == sensor and item[1] == variable:
                    self._state.pop(i)
                    break

    # 5. Rehacer el cambio deshecho
    def redo(self):
        if self._redo_stack.is_empty():
            raise Empty("No hay cambios que rehacer")

        sensor, variable, existed_before, previous_value, current_value = self._redo_stack.pop()
        self._stack.push((sensor, variable, existed_before, previous_value, current_value))

        index_to_update = -1
        for i, item in enumerate(self._state):
            if item[0] == sensor and item[1] == variable:
                index_to_update = i
                break

        if index_to_update != -1:
            self._state[index_to_update] = (sensor, variable, current_value)
        else:
            self._state.append((sensor, variable, current_value))

    # 6. Registros pendientes
    def pending(self):
        return len(self._queue)

    # 7. Consulta de valor actual
    def current_value(self, sensor, variable):
        for item in self._state:
            if item[0] == sensor and item[1] == variable:
                return item[2]
        raise KeyError(f"No existe un valor para el sensor '{sensor}' y variable '{variable}'")
